# 1001 22일차

## 1. 임베딩

자연어를 모델이 연산할 수 있는 실수 벡터로 바꾸는 것

| 단계 | 예 |
|---|---|
| 문장 | 나는 밥을 |
| 토큰화 | `[나는, 밥을]` |
| 정수 인코딩 | `[3, 17]` |
| 임베딩 | `[[0.21, -0.53, ...], [0.88, 0.10, ...]]` |

- 수치화는 정수 인코딩에서 이미 됨. 임베딩은 그다음 단계
- 번호는 이름표라 뜻이 비슷한지 알 수 없음
- 임베딩 값은 가중치라 학습하면서 뜻이 비슷한 단어끼리 가까워짐

### 1-1. 원핫의 한계

단어 하나가 단어 종류 수만큼의 칸이 되고 그중 1칸만 1

| | 원핫 | 임베딩 `Embedding(31, 10)` |
|---|---|---|
| 모양 | `(15, 5, 31)` | `(15, 5, 10)` |
| 전체 칸 | 2325 | 750 |
| 0인 칸 | 2250 (약 97%) | 고정된 0 없음 |

1. 칸 수 : 단어가 수만 개면 단어 하나가 수만 칸
2. 거리 : 모든 단어 사이 거리가 같음

- 패딩의 0은 임베딩을 써도 남음. 0번 벡터로 바뀔 뿐

### 1-2. Embedding 층

단어 번호마다 벡터를 한 줄씩 가진 표. 번호로 그 줄을 꺼냄

![패딩한 문장 [0, 0, 0, 2, 3]의 번호마다 30줄 10칸짜리 표에서 같은 번호의 줄을 꺼낸다. 0 세 칸은 모두 0번 줄을 꺼내고, 꺼낸 줄을 쌓으면 (5, 10)이 된다. 파라미터는 표의 칸 수 300개다](assets/embedding-lookup.svg)

```python
Embedding(input_dim=30, output_dim=10, input_length=5)   # (15, 5) > (15, 5, 10)
```

1. `input_dim` : 표의 줄 수. 단어 사전 크기
2. `output_dim` : 한 줄의 길이. 단어 하나를 몇 칸 벡터로
3. `input_length` : 문장 길이

- 파라미터 = 표의 칸 수 = `input_dim × output_dim = 300`. `input_length`와 bias는 없음
- 첫 인자가 출력 크기인 다른 층과 달리 입력(사전 크기)이 먼저
- `input_length`는 이름을 붙여 씀. 세 번째 자리는 `embeddings_initializer`
- `input_length`를 빼면 출력이 `(None, None, 10)`. 파라미터는 같음
- 같은 번호는 언제나 같은 벡터. 값은 범위 제한 없는 가중치

## 2. Tokenizer

문장을 토큰으로 자르고 번호를 붙이는 케라스 도구

1. `fit_on_texts` : 단어 사전을 만듦
2. `word_index` : 단어별 번호
3. `word_counts` : 단어별 등장 횟수
4. `texts_to_sequences` : 문장을 번호 목록으로 바꿈

- 많이 나온 단어가 앞 번호. 같으면 먼저 나온 순서. 번호는 1부터
- 소문자로 바꾸고 문장부호를 지움. `it's`는 한 단어
- 여러 문장을 넣으면 사전 하나에 합쳐서 셈
- 번호를 그대로 넣으면 모델이 크기로 계산함. 그래서 원핫이나 임베딩을 거침

### 2-1. 원핫 전 모양

단어마다 원핫하려면 단어가 샘플 축(맨 앞)에 와야 함. `texts_to_sequences` 결과는 문장 하나여도 `(1, 16)`

| 도구 | 받는 모양 | 넣기 전 | 결과 |
|---|---|---|---|
| `pd.get_dummies` | 1차원 | `(16,)` | `(16, 13)` |
| `OneHotEncoder` | 2차원 | `(16, 1)` | `(16, 13)` |
| `to_categorical` | 아무 차원 | `(1, 16)` 그대로 | `(1, 16, 14)` |

- `(1, 16)`을 `OneHotEncoder`에 넣으면 에러 없이 샘플 1개로 읽힘
- `to_categorical`은 0부터 칸을 만듦. 번호가 1부터면 0번 열이 빔

## 3. 패딩

문장 길이를 0으로 채우거나 잘라서 맞추는 것. 길이가 다르면 배열을 만들 수 없음

![pre는 [0, 0, 0, 2, 3]이라 실제 단어를 마지막에 읽고 바로 출력하고, post는 [2, 3, 0, 0, 0]이라 실제 단어를 읽은 뒤 0을 세 칸 더 읽고 출력한다](assets/padding-pre-post.svg)

1. `padding='pre'` : 앞을 0으로 채움. 기본값
2. `padding='post'` : 뒤를 0으로 채움
3. `maxlen` : 맞출 길이
4. `truncating` : 긴 문장을 자를 쪽. 기본값 `'pre'`

- 0을 쓸 수 있는 이유 : `Tokenizer` 번호가 1부터라 0번이 비어 있음
- 앞에 채우는 이유 : RNN은 마지막 칸 결과를 내보냄. 뒤에 채우면 실제 단어 뒤에 0을 더 읽어 앞 단어의 영향이 줄어듦 (day18 §3-4)
- 예측할 문장도 같은 `maxlen`, `padding`으로